# Intertemporal choice: Random Forest (standalone notebook)

**Self-contained:** uses only **`clean_data_basic.csv`** (same base table as `choice_model_readable.ipynb` / `data_cleaning.py`). No other teammate-specific cleaning scripts.

## What the model predicts

The model predicts whether a participant chooses the **patient** option (wait for a larger later reward, **LL**) or the **impatient** option (take the smaller sooner reward, **SS**), on **held-out subjects** the model never trained on.

## Why Random Forest (alongside logistic regression)

| Idea | What to say |
|------|-------------|
| **Nonlinear patterns** | Trees can capture thresholds and interactions between amounts, delays, and context without writing every interaction by hand. |
| **Interpretability** | Feature importances and optional SHAP support a clear narrative; pair with the logistic notebook for coefficients / odds ratios. |
| **Tabular baseline** | Strong default on mixed numeric features after imputation. |
| **Same evaluation rule** | **Subject-level** train/test so trials from the same person are not split across train and test. |

## How to read the metrics (plain language)

Use the **numbers printed after you run** this notebook (they depend on the exact CSV and seed).

- **Accuracy** — Fraction of **test trials** where predicted choice matches the real choice. You can say: *“the model got about X out of 10 trial-level predictions right on people it never trained on.”*
- **ROC-AUC** — How well predicted probabilities **separate** SS vs LL trials; `0.5` is random, `1.0` is perfect ranking. It is **not** the same as accuracy.
- **Precision (LL)** — When the model predicts **patient / LL**, how often that was correct.
- **Recall (LL)** — Among trials that were truly **LL**, what fraction the model labeled as LL.

**Method note:** `patience_rate` (subject-level average choice) is built **only from training subjects’ training rows**, then applied to train and test. The old pattern of computing it on the **full** dataframe before splitting would **leak** test labels into training.

## Requirements

`pandas`, `numpy`, `scikit-learn`, `matplotlib`. Optional: `shap` if `RUN_SHAP = True`.

## How to use

1. Place `clean_data_basic.csv` next to this notebook (or set an absolute `CSV_PATH`).
2. Edit **Configuration**, then **Run all**.

In [ ]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [ ]:
# Paths & run knobs
CSV_PATH = Path("clean_data_basic.csv")
OUT_DIR = Path("results_rf")  # None to skip saving files
NROWS = None  # None = full file; e.g. 100_000 for speed
TEST_SIZE_SUBJECTS = 0.2
SEED = 42

RF_N_ESTIMATORS = 200
RF_MAX_DEPTH = 10
RF_MIN_SAMPLES_LEAF = 50
RF_CLASS_WEIGHT = "balanced"

RUN_LOGISTIC_BASELINE = True
RUN_SHAP = False
SHAP_SAMPLE = 2000

## Pipeline code (load → features → split → train → evaluate)

In [ ]:
from typing import Optional, Union

# Columns to read (intersection with file header — keeps I/O light)
BASE_READ_COLS = [
    "choice",
    "subj_ident",
    "ss_value",
    "ss_time",
    "ll_value",
    "ll_time",
    "rt",
    "trial_idx",
    "time_pressure",
    "online_study",
    "incentivization",
    "value_diff",
    "time_diff_days",
]


def load_clean_csv(path: Path, nrows: Optional[int]) -> pd.DataFrame:
    t0 = time.perf_counter()
    header = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    usecols = [c for c in BASE_READ_COLS if c in header]
    if "choice" not in usecols or "subj_ident" not in usecols:
        raise ValueError(f"CSV missing choice/subj_ident. Available: {usecols}")
    kw = dict(usecols=usecols, low_memory=False, engine="c")
    if nrows is not None:
        kw["nrows"] = nrows
    df = pd.read_csv(path, **kw)
    print(f"Loaded {len(df):,} rows × {df.shape[1]} columns in {time.perf_counter() - t0:.1f}s")
    return df


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    # Row-level features only (no labels from other splits).
    out = df.copy()
    ss = pd.to_numeric(out["ss_value"], errors="coerce")
    ll = pd.to_numeric(out["ll_value"], errors="coerce")
    out["reward_ratio"] = np.where(ss.fillna(0) > 0, ll / ss, np.nan)
    out["delay_diff"] = pd.to_numeric(out["ll_time"], errors="coerce") - pd.to_numeric(
        out["ss_time"], errors="coerce"
    )
    if "time_diff_days" in out.columns:
        out["delay_diff"] = out["delay_diff"].fillna(pd.to_numeric(out["time_diff_days"], errors="coerce"))
    rt = pd.to_numeric(out["rt"], errors="coerce")
    out["rt_log"] = np.log(rt.where(rt > 0))
    if "incentivization" in out.columns:
        inc = out["incentivization"].astype("string")
    else:
        inc = pd.Series("__na__", index=out.index, dtype="string")
    out["incentivization_code"] = pd.factorize(inc.fillna("__na__"))[0]
    D = out["delay_diff"]
    ratio = out["reward_ratio"]
    k_thr = (ratio - 1) / D.where(D > 0)
    out["log_discount_k"] = -np.log(k_thr.where(k_thr > 0))
    return out


def add_patience_rate_no_leakage(df: pd.DataFrame, train_subjects: np.ndarray) -> pd.DataFrame:
    # Subject mean choice from TRAINING subjects only (train rows) — avoids test-label leakage.
    out = df.copy()
    tr_mask = out["subj_ident"].isin(train_subjects)
    train_rows = out.loc[tr_mask]
    subj_mean = train_rows.groupby("subj_ident", sort=False)["choice"].mean()
    global_mean = float(train_rows["choice"].mean())
    out["patience_rate"] = out["subj_ident"].map(subj_mean).fillna(global_mean)
    return out


def preferred_feature_columns(df: pd.DataFrame) -> list[str]:
    cols = [
        "ss_value",
        "ss_time",
        "ll_value",
        "ll_time",
        "rt",
        "trial_idx",
        "reward_ratio",
        "delay_diff",
        "rt_log",
        "log_discount_k",
        "patience_rate",
        "time_pressure",
        "online_study",
        "incentivization_code",
    ]
    return [c for c in cols if c in df.columns]


def run_random_forest_pipeline(
    *,
    csv_path: Path,
    out_dir: Optional[Path],
    nrows: Optional[int],
    test_size_subjects: float,
    seed: int,
    rf_n_estimators: int,
    rf_max_depth: int,
    rf_min_samples_leaf: int,
    rf_class_weight: Optional[Union[str, dict]],
    run_logistic_baseline: bool,
    run_shap: bool,
    shap_sample: int,
) -> None:
    lr_pred = None
    df = load_clean_csv(csv_path, nrows)
    df["choice"] = pd.to_numeric(df["choice"], errors="coerce")
    df = df[df["choice"].isin([0, 1])].copy()
    df["choice"] = df["choice"].astype(int)
    df["subj_ident"] = df["subj_ident"].astype(str).str.strip()

    df = add_engineered_features(df)

    subjects = df["subj_ident"].unique()
    train_subj, test_subj = train_test_split(
        subjects, test_size=test_size_subjects, random_state=seed
    )
    train_subj = np.asarray(train_subj)
    test_subj = np.asarray(test_subj)

    df = add_patience_rate_no_leakage(df, train_subj)

    feature_cols = preferred_feature_columns(df)
    need = feature_cols + ["choice"]
    before = len(df)
    df = df.dropna(subset=need)
    print(f"Rows after dropna on model features: {len(df):,} (dropped {before - len(df):,})")

    train_mask = df["subj_ident"].isin(train_subj)
    test_mask = df["subj_ident"].isin(test_subj)
    train = df.loc[train_mask]
    test = df.loc[test_mask]

    X_train = train[feature_cols]
    y_train = train["choice"]
    X_test = test[feature_cols]
    y_test = test["choice"]

    n_test_trials = len(y_test)
    print(f"\nTrain trials: {len(y_train):,} | Test trials: {n_test_trials:,}")
    print(f"Train subjects: {train['subj_ident'].nunique():,} | Test subjects: {test['subj_ident'].nunique():,}")

    rf_pipe = Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            (
                "rf",
                RandomForestClassifier(
                    n_estimators=rf_n_estimators,
                    max_depth=rf_max_depth,
                    min_samples_leaf=rf_min_samples_leaf,
                    n_jobs=-1,
                    random_state=seed,
                    class_weight=rf_class_weight,
                ),
            ),
        ]
    )
    t1 = time.perf_counter()
    rf_pipe.fit(X_train, y_train)
    print(f"RF fit time: {time.perf_counter() - t1:.1f}s")

    rf_pred = rf_pipe.predict(X_test)
    rf_proba = rf_pipe.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, rf_pred)
    auc = roc_auc_score(y_test, rf_proba)
    prec = precision_score(y_test, rf_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, rf_pred, pos_label=1, zero_division=0)

    print("\n" + "=" * 60)
    print("RANDOM FOREST — held-out subjects (test trials never seen in training)")
    print("=" * 60)
    print(
        f"The model answers the SS-vs-LL question {n_test_trials:,} times on test trials "
        f"(people entirely held out from training, subject-level split)."
    )
    print(f"\nAccuracy:  {acc:.4f}  (~{acc*100:.1f}% of test trial predictions correct)")
    print(f"ROC-AUC:   {auc:.4f}  (1.0 = perfect ranking, 0.5 = random)")
    print(f"Precision (LL / 'patient'): {prec:.4f}  — when it predicts LL, how often correct")
    print(f"Recall (LL):                {rec:.4f}  — share of true LL trials caught")
    print("\n" + classification_report(y_test, rf_pred, target_names=["SS (impatient)", "LL (patient)"]))

    if run_logistic_baseline:
        imputer = SimpleImputer(strategy="median")
        X_tr_i = imputer.fit_transform(X_train)
        X_te_i = imputer.transform(X_test)
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr_i)
        X_te_sc = scaler.transform(X_te_i)
        lr = LogisticRegression(max_iter=2000, random_state=seed)
        lr.fit(X_tr_sc, y_train)
        lr_pred = lr.predict(X_te_sc)
        lr_proba = lr.predict_proba(X_te_sc)[:, 1]
        print("\n── Logistic baseline (same split, median impute + scale) ──")
        print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}  ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")

    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)

    rf_model = rf_pipe.named_steps["rf"]
    imp_df = (
        pd.DataFrame({"feature": feature_cols, "importance": rf_model.feature_importances_})
        .sort_values("importance", ascending=True)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(imp_df["feature"], imp_df["importance"], color="steelblue", edgecolor="white")
    ax.set_xlabel("Importance (Gini decrease, approximate)")
    ax.set_title("Random Forest — feature importance (clean_data_basic.csv)", fontweight="bold")
    fig.tight_layout()
    if out_dir is not None:
        fig.savefig(out_dir / "plot_rf_importance.png", bbox_inches="tight", dpi=150)
    plt.show()

    if run_logistic_baseline and lr_pred is not None:
        fig2, axes = plt.subplots(1, 2, figsize=(12, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, lr_pred, ax=axes[0], display_labels=["SS (impatient)", "LL (patient)"], colorbar=False
        )
        axes[0].set_title("Logistic regression", fontweight="bold")
        ConfusionMatrixDisplay.from_predictions(
            y_test, rf_pred, ax=axes[1], display_labels=["SS (impatient)", "LL (patient)"], colorbar=False
        )
        axes[1].set_title("Random Forest", fontweight="bold")
    else:
        fig2, ax_cm = plt.subplots(figsize=(6, 5))
        axes = [ax_cm]
        ConfusionMatrixDisplay.from_predictions(
            y_test, rf_pred, ax=ax_cm, display_labels=["SS (impatient)", "LL (patient)"], colorbar=False
        )
        ax_cm.set_title("Random Forest", fontweight="bold")
    fig2.suptitle("Confusion matrices — held-out subjects", fontsize=13, fontweight="bold")
    fig2.tight_layout()
    if out_dir is not None:
        fig2.savefig(out_dir / "plot_confusion_matrix.png", bbox_inches="tight", dpi=150)
    plt.show()

    if run_shap:
        try:
            import shap
        except ImportError:
            print("SHAP not installed; pip install shap or set RUN_SHAP = False")
        else:
            X_samp = X_test.sample(n=min(shap_sample, len(X_test)), random_state=seed)
            explainer = shap.TreeExplainer(rf_model)
            shap_values = explainer.shap_values(X_samp)
            if isinstance(shap_values, list):
                shap_ll = np.asarray(shap_values[1])
            else:
                sv = np.asarray(shap_values)
                shap_ll = sv[..., 1] if sv.ndim == 3 else sv
            plt.figure(figsize=(10, 6))
            shap.summary_plot(
                shap_ll, X_samp, feature_names=feature_cols, plot_type="bar", show=False
            )
            plt.title("SHAP (RF) — impact on LL probability", fontweight="bold")
            plt.tight_layout()
            if out_dir is not None:
                plt.savefig(out_dir / "plot_shap_bar.png", bbox_inches="tight", dpi=150)
            plt.show()

    metrics = {
        "csv": str(csv_path.resolve()),
        "n_test_trials": int(n_test_trials),
        "n_test_subjects": int(test["subj_ident"].nunique()),
        "accuracy": float(acc),
        "roc_auc": float(auc),
        "precision_ll": float(prec),
        "recall_ll": float(rec),
        "features": feature_cols,
    }
    if out_dir is not None:
        with open(out_dir / "rf_metrics.json", "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)
        print(f"\nSaved metrics + plots under: {out_dir.resolve()}")

## Run

In [ ]:
run_random_forest_pipeline(
    csv_path=CSV_PATH,
    out_dir=OUT_DIR,
    nrows=NROWS,
    test_size_subjects=TEST_SIZE_SUBJECTS,
    seed=SEED,
    rf_n_estimators=RF_N_ESTIMATORS,
    rf_max_depth=RF_MAX_DEPTH,
    rf_min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    rf_class_weight=RF_CLASS_WEIGHT,
    run_logistic_baseline=RUN_LOGISTIC_BASELINE,
    run_shap=RUN_SHAP,
    shap_sample=SHAP_SAMPLE,
)